In [ ]:
import numpy as np
import re
re = re.compile(r'[0-9.]+')
a = np.loadtxt('2/m.csv',[('f','3B'),('df','3B'),('x','3B'),('d','3B'),('t','3B'),('b','3B'),('_','f')],delimiter=',',converters=lambda x:next(re.finditer(x)).group()if x!='""' else 0,skiprows=1,usecols=(4,5,6,7,8,0,9,10,11,12,13,0,22,23,24,25,26,27,33),encoding='utf-8-sig').view(np.recarray)
a = a[a._.astype('?')]
a.df[:,2] = 3 - a.df[:,:2].sum(1)
a.d[:,2] = 3 - a.d[:,:2].sum(1)
a.t[:,0] >>= 5
a._ = 1 / a._
N = np.arange(len(a), dtype='H')[:,None]
df_ = a.df[:,0]*2+a.df[:,1]-(a.df[:,1]>a.df[:,0])
d_ = a.d[:,0]*2+a.d[:,1]-(a.d[:,1]>a.d[:,0])
fi = a.f[N,a.df]
i = a.x[N,a.d]
dfd = df_*6+d_
reg = (i[:,1]+1)*i[:,2]+1
regm = (np.sort(a.x,1)[:,1]+1)*a.x.min(1)+1
regr = np.divide(reg, regm, dtype='f')
a = np.rec.fromarrays((a,df_,fi.view('V3')[:,0],d_,i.view('V3')[:,0],dfd,reg,regm,regr)).view([('f','3B'),('df','3B'),('x','3B'),('d','3B'),('t','3B'),('b','3B'),('_','f'),('df_','B'),('fi','3B'),('d_','B'),('i','3B'),('dfd','B'),('reg','B'),('regm','B'),('regr','f')])
a.sort(order='_')
print(a[-10:])

t = a[(a.fi[:,:2]==1).all(1)]
t.df[:,:2] = t.df[:,1::-1]
a = np.concatenate((a,t)).view(np.recarray)

t = a[(a.f==1).all(1)]
T = t.copy()
t.df[:,::2] = t.df[:,::-2]
T.df[:,1:] = T.df[:,:0:-1]
a = np.concatenate((a,t,T)).view(np.recarray)

t = a[(a.i[:,2:]==1).all(1)]
t.d[:,1:] = t.d[:,:0:-1]
a = np.concatenate((a,t)).view(np.recarray)

t = a[(a.x==1).all(1)]
T = t.copy()
t.d[:,::2] = t.d[:,::-2]
T.d[:,:2] = T.d[:,1::-1]
a = np.concatenate((a,t,T)).view(np.recarray)

In [ ]:
for i in a.dtype.fields:
    if i in ('_', 'df', 'd'):
        continue
    t = a[i]
    if t.ndim == 1:
        t = t[:,None]
    print(i)
    plt.figure(figsize=(t.shape[1], 1))
    plt.subplots_adjust(0, 0, 1, 1, 0, 0)
    for j, J in enumerate(t.T):
        plt.subplot(1, t.shape[1], j + 1).scatter(J, a._, 1)
    plt.show()